# P01b — Structural Integration

Build the master country reference and coverage matrix. Ensure all Phase 0b datasets join smoothly to QoG's country spine.

**Goal:** After this notebook, every dataset joins on `ident_ccodealp` without per-dataset remapping. Coverage is queryable per country × year × dataset.

In [1]:
using DataFrames, Arrow, CSV, Statistics

## 1. Load All Data Sources

Load the QoG spine and all Phase 0b Arrow files. Inventory what we have.

In [2]:
# QoG augmented (authoritative spine)
qog = DataFrame(Arrow.Table("data/qog_std_ts_jan25_aug.arrow"))
println("QoG: $(nrow(qog)) rows, $(length(unique(skipmissing(qog.ident_ccodealp)))) countries")

# Phase 0b datasets
emdat_events = DataFrame(Arrow.Table("data/emdat_events.arrow"))
emdat_cy = DataFrame(Arrow.Table("data/emdat_country_year.arrow"))
println("EM-DAT: $(nrow(emdat_events)) events, $(nrow(emdat_cy)) country-years, $(length(unique(emdat_cy.iso3))) countries")

dose_sub = DataFrame(Arrow.Table("data/dose_subnational.arrow"))
dose_nat = DataFrame(Arrow.Table("data/dose_national.arrow"))
println("DOSE: $(nrow(dose_sub)) subnational, $(nrow(dose_nat)) national, $(length(unique(dose_sub.iso3))) countries")

shdi = DataFrame(Arrow.Table("data/shdi_v10.arrow"))
println("SHDI: $(nrow(shdi)) rows, $(length(unique(shdi.isocode3))) countries")

geodist = DataFrame(Arrow.Table("data/cepii_geodist.arrow"))
geo_countries = DataFrame(Arrow.Table("data/cepii_geo_countries.arrow"))
println("GeoDist: $(nrow(geodist)) pairs, $(nrow(geo_countries)) countries")

gravity = DataFrame(Arrow.Table("data/cepii_gravity.arrow"))
gravity_countries = DataFrame(Arrow.Table("data/cepii_gravity_countries.arrow"))
println("Gravity: $(nrow(gravity)) rows, $(nrow(gravity_countries)) countries")

lv_panel = DataFrame(Arrow.Table("data/lv_crisis_panel.arrow"))
lv_outcomes = DataFrame(Arrow.Table("data/lv_outcomes.arrow"))
println("L&V: $(nrow(lv_panel)) crisis-years, $(nrow(lv_outcomes)) episodes, $(length(unique(lv_panel.iso3))) countries")

QoG: 12391 rows, 202 countries
EM-DAT: 26660 events, 6919 country-years, 231 countries
DOSE: 46851 subnational, 2092 national, 83 countries
SHDI: 65031 rows, 188 countries
GeoDist: 50176 pairs, 238 countries
Gravity: 4572288 rows, 252 countries
L&V: 463 crisis-years, 151 episodes, 118 countries


## 2. ISO3 Key Inventory

What column does each dataset use for country codes? Are they all joinable to QoG's `ident_ccodealp`?

In [3]:
# ISO3 column names and types across all datasets
qog_codes = Set(unique(skipmissing(qog.ident_ccodealp)))

datasets = [
    ("QoG",        qog_codes,                                             "ident_ccodealp"),
    ("EM-DAT",     Set(unique(emdat_cy.iso3)),                            "iso3"),
    ("DOSE",       Set(unique(dose_sub.iso3)),                            "iso3"),
    ("SHDI",       Set(unique(shdi.isocode3)),                            "isocode3  ← MISMATCH"),
    ("GeoDist",    Set(unique(skipmissing(geodist.iso_o))),               "iso_o / iso_d"),
    ("Gravity",    Set(unique(skipmissing(gravity.iso3_o))),              "iso3_o / iso3_d"),
    ("L&V",        Set(unique(skipmissing(lv_panel.iso3))),               "iso3"),
]

println("Dataset          Countries  Key Column            Matched to QoG")
println("─"^75)
for (name, codes, col) in datasets
    matched = length(intersect(codes, qog_codes))
    println("$(rpad(name, 18))$(lpad(string(length(codes)), 5))      $(rpad(col, 22)) $matched / $(length(qog_codes))")
end

Dataset          Countries  Key Column            Matched to QoG
───────────────────────────────────────────────────────────────────────────
QoG                 202      ident_ccodealp         202 / 202
EM-DAT              231      iso3                   196 / 202
DOSE                 83      iso3                   82 / 202
SHDI                188      isocode3  ← MISMATCH   186 / 202
GeoDist             224      iso_o / iso_d          190 / 202
Gravity             243      iso3_o / iso3_d        201 / 202
L&V                 118      iso3                   118 / 202


## 3. Build Master Country Reference

QoG spine + geographic lookup + Phase 1 status + existence dates + coverage flags.

In [4]:
# Step 1: QoG country spine — one row per country
qog_spine = combine(groupby(dropmissing(qog, :ident_ccodealp), :ident_ccodealp),
    :ident_cname => first => :ident_cname,
    :ident_ccode => first => :ident_ccode,
    :ident_year => minimum => :qog_first_year,
    :ident_year => maximum => :qog_last_year,
    :ident_year => length => :qog_n_years
)
println("QoG spine: $(nrow(qog_spine)) countries")
first(qog_spine, 5)

QoG spine: 202 countries


Row,ident_ccodealp,ident_cname,ident_ccode,qog_first_year,qog_last_year,qog_n_years
,String,String,Int64,Int64,Int64,Int64
1,AFG,Afghanistan,4,1946,2024,79
2,ALB,Albania,8,1946,2024,79
3,DZA,Algeria,12,1963,2024,62
4,AND,Andorra,20,1946,2024,79
5,AGO,Angola,24,1976,2024,49


In [5]:
# Step 2: Join geographic lookup (UN regions, continents)
geo_lookup = CSV.read("data/ggis_geographic_lookup.csv", DataFrame)
geo_cols = select(geo_lookup, 
    :ident_ccodealp, 
    :un_region_name => :un_region,
    :un_subregion_name => :un_subregion,
    :continent
)

master = leftjoin(qog_spine, geo_cols, on=:ident_ccodealp)
println("After geo join: $(nrow(master)) rows, $(count(!ismissing, master.un_region)) with UN region")

# Check who's missing UN region
missing_geo = filter(r -> ismissing(r.un_region), master)
if nrow(missing_geo) > 0
       println("Missing UN region: $(missing_geo.ident_ccodealp)")
end

After geo join: 202 rows, 198 with UN region
Missing UN region: ["SCG", "VDR", "YUG", "XTI"]


In [6]:
# Check for dups in geo_cols
println("Geo lookup rows: $(nrow(geo_cols))")
geo_dups = combine(groupby(geo_cols, :ident_ccodealp), nrow => :n)
filter!(r -> r.n > 1, geo_dups)
println("Geo duplicates: $(geo_dups)")


Geo lookup rows: 206
Geo duplicates: 0×2 DataFrame
 Row │ ident_ccodealp  n
     │ String7         Int64
─────┴───────────────────────


In [9]:
# Step 3: Compute coverage flags — which datasets have data for each country?
emdat_countries = Set(unique(emdat_cy.iso3))
dose_countries = Set(unique(dose_sub.iso3))
shdi_countries = Set(unique(shdi.isocode3))  # Note: isocode3, not iso3
geodist_countries = Set(unique(skipmissing(geodist.iso_o)))
gravity_countries_set = Set(unique(skipmissing(gravity.iso3_o)))
lv_countries = Set(unique(skipmissing(lv_panel.iso3)))

master[!, :has_emdat] = [c in emdat_countries for c in master.ident_ccodealp]
master[!, :has_dose] = [c in dose_countries for c in master.ident_ccodealp]
master[!, :has_shdi] = [c in shdi_countries for c in master.ident_ccodealp]
master[!, :has_geodist] = [c in geodist_countries for c in master.ident_ccodealp]
master[!, :has_gravity] = [c in gravity_countries_set for c in master.ident_ccodealp]
master[!, :has_lv] = [c in lv_countries for c in master.ident_ccodealp]

# Summary
println("Coverage flags:")
for col in [:has_emdat, :has_dose, :has_shdi, :has_geodist, :has_gravity, :has_lv]
    n = count(master[!, col])
    pct = round(100 * n / nrow(master), digits=1)
    println("  $(rpad(string(col), 15)) $n / $(nrow(master))  ($pct%)")
end

Coverage flags:
  has_emdat       196 / 202  (97.0%)
  has_dose        82 / 202  (40.6%)
  has_shdi        186 / 202  (92.1%)
  has_geodist     190 / 202  (94.1%)
  has_gravity     201 / 202  (99.5%)
  has_lv          118 / 202  (58.4%)


In [10]:
# Step 4: Dataset-specific existence dates
# Compute first/last year each country appears in each external dataset
function year_range(df, iso_col, year_col)
    valid = dropmissing(df, [iso_col, year_col])
    combine(groupby(valid, iso_col),
        year_col => minimum => :first_year,
        year_col => maximum => :last_year
    )
end

emdat_years = year_range(emdat_cy, :iso3, :year)
dose_years = year_range(dose_sub, :iso3, :year)
shdi_years = year_range(shdi, :isocode3, :year)
gravity_years = year_range(gravity, :iso3_o, :year)

# Rename for joining
for (df, prefix, key) in [
    (emdat_years, "emdat", :iso3),
    (dose_years, "dose", :iso3),
    (shdi_years, "shdi", :isocode3),
    (gravity_years, "gravity", :iso3_o)
]
    rename!(df, :first_year => Symbol("$(prefix)_first_year"), :last_year => Symbol("$(prefix)_last_year"))
    rename!(df, key => :ident_ccodealp)
end

# Join all to master
for df in [emdat_years, dose_years, shdi_years, gravity_years]
    master = leftjoin(master, df, on=:ident_ccodealp)
end

println("Master with existence dates: $(nrow(master)) rows × $(ncol(master)) cols")
println("Columns: $(names(master))")

Master with existence dates: 202 rows × 23 cols
Columns: ["ident_ccodealp", "ident_cname", "ident_ccode", "qog_first_year", "qog_last_year", "qog_n_years", "un_region", "un_subregion", "continent", "has_emdat", "has_dose", "has_shdi", "has_geodist", "has_gravity", "has_lv", "emdat_first_year", "emdat_last_year", "dose_first_year", "dose_last_year", "shdi_first_year", "shdi_last_year", "gravity_first_year", "gravity_last_year"]


In [11]:
# Inspect the master — spot check a few countries
for c in ["USA", "AFG", "DEU", "SSD", "TWN", "MCO"]
    row = filter(r -> r.ident_ccodealp == c, master)
    if nrow(row) > 0
        r = first(row)
        datasets = join([col for col in [:has_emdat, :has_dose, :has_shdi, :has_geodist, :has_gravity, :has_lv] if r[col]], ", ")
        println("$(rpad(c, 5)) $(rpad(r.ident_cname, 25)) QoG $(r.qog_first_year)-$(r.qog_last_year)  datasets: $datasets")
    end
end

USA   United States of America (the) QoG 1946-2024  datasets: has_emdat, has_dose, has_shdi, has_geodist, has_gravity, has_lv
AFG   Afghanistan               QoG 1946-2024  datasets: has_emdat, has_shdi, has_geodist, has_gravity
DEU   Germany                   QoG 1949-2024  datasets: has_emdat, has_dose, has_shdi, has_geodist, has_gravity, has_lv
SSD   South Sudan               QoG 2011-2024  datasets: has_emdat, has_shdi, has_gravity
TWN   Taiwan (Province of China) QoG 1950-2024  datasets: has_emdat, has_geodist, has_gravity
MCO   Monaco                    QoG 1946-2024  datasets: has_gravity


In [12]:
# Top 30 by population — use QoG's most recent year with WDI population
qog_recent = filter(r -> !ismissing(r.ident_ccodealp) && r.ident_year == 2022, qog)

# Find the population slug
pop_col = :wdi_pop  # WDI total population
if hasproperty(qog_recent, pop_col)
    pop_df = select(dropmissing(qog_recent, pop_col), :ident_ccodealp, :ident_cname, pop_col)
    sort!(pop_df, pop_col, rev=true)
    top30_pop = first(pop_df, 30)
    
    println("Top 30 by population (2022) — dataset coverage:")
    println("─"^110)
    for r in eachrow(top30_pop)
        m = filter(row -> row.ident_ccodealp == r.ident_ccodealp, master)
        nrow(m) == 0 && continue
        mr = first(m)
        flags = join([replace(string(col), "has_" => "") for col in [:has_emdat, :has_dose, :has_shdi, :has_geodist, :has_gravity, :has_lv] if mr[col]], ", ")
        missing_flags = join([replace(string(col), "has_" => "") for col in [:has_emdat, :has_dose, :has_shdi, :has_geodist, :has_gravity, :has_lv] if !mr[col]], ", ")
        pop_m = round(r[pop_col] / 1e6, digits=1)
        println("  $(rpad(r.ident_ccodealp, 5)) $(rpad(r.ident_cname, 35)) $(lpad(string(pop_m), 8))M  has: $flags$(isempty(missing_flags) ? "" : "  MISSING: $missing_flags")")
    end
else
    println("Population column $pop_col not found. Available wdi columns:")
    for n in names(qog_recent)
        occursin("pop", lowercase(string(n))) && println("  $n")
    end
end


Top 30 by population (2022) — dataset coverage:
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  IND   India                                 1417.2M  has: emdat, dose, shdi, geodist, gravity, lv
  CHN   China                                 1412.2M  has: emdat, dose, shdi, geodist, gravity, lv
  USA   United States of America (the)         333.3M  has: emdat, dose, shdi, geodist, gravity, lv
  IDN   Indonesia                              275.5M  has: emdat, dose, shdi, geodist, gravity, lv
  PAK   Pakistan                               235.8M  has: emdat, dose, shdi, geodist, gravity  MISSING: lv
  NGA   Nigeria                                218.5M  has: emdat, dose, shdi, geodist, gravity, lv
  BRA   Brazil                                 215.3M  has: emdat, dose, shdi, geodist, gravity, lv
  BGD   Bangladesh                             171.2M  has: emdat, shdi, geodist, gravity, lv  MISSING: dose
  RUS   Russian Federat

In [13]:
# Top 30 by GDP — use WDI GDP
gdp_col = :wdi_gdpcapcon2015  # GDP per capita constant 2015 USD — need total GDP
gdp_total_col = :wdi_gdp  # or similar

# Find the right GDP column
gdp_candidates = [n for n in names(qog_recent) if occursin("gdp", lowercase(string(n))) && !occursin("cap", lowercase(string(n)))]
println("GDP candidates: $gdp_candidates")


GDP candidates: ["eu_eco2gdpeurhab", "eu_eco2gdpmioeur", "gain_gaingdp", "gain_readgdp", "gain_vulngdp", "gle_cgdpc", "gle_gdp", "gle_rgdpc", "gted_rfgdp", "gtr_centaxgdp", "gtr_centaxgdp1800", "gtr_centaxgdp1850", "gtr_centaxgdp1900", "mad_gdppc", "mad_gdppc1000", "mad_gdppc1300", "mad_gdppc1400", "mad_gdppc1500", "mad_gdppc1600", "mad_gdppc1700", "mad_gdppc1800", "mad_gdppc1900", "oecd_evogdp_t1", "oecd_sizegdp_t1", "oecd_tradegdp_t1a", "oecd_tradegdp_t1b", "pwt_rgdp", "pwt_slcgdp", "rd_inw_gdp", "wdi_chexppgdp", "wdi_gdpagr", "wdi_gdpgr", "wdi_gdpind", "wdi_gdppppcon2021", "wdi_gdppppcur", "wdi_svapgdp"]


In [16]:
# Check which GDP columns have data for 2022
for col in [:gle_gdp, :wdi_gdppppcon2021, :pwt_rgdp]
    if hasproperty(qog_recent, col)
        n = count(!ismissing, qog_recent[!, col])
        println("$col: $n / $(nrow(qog_recent)) non-missing for 2022")
    end
end


gle_gdp: 0 / 194 non-missing for 2022
wdi_gdppppcon2021: 181 / 194 non-missing for 2022
pwt_rgdp: 0 / 194 non-missing for 2022


In [17]:
# Use 2020 — most sources have data
qog_2020 = filter(r -> !ismissing(r.ident_ccodealp) && r.ident_year == 2020, qog)
for col in [:gle_gdp, :wdi_gdppppcon2021, :pwt_rgdp]
    if hasproperty(qog_2020, col)
        n = count(!ismissing, qog_2020[!, col])
        println("$col (2020): $n non-missing")
    end
end


gle_gdp (2020): 0 non-missing
wdi_gdppppcon2021 (2020): 183 non-missing
pwt_rgdp (2020): 0 non-missing


In [18]:
# Top 30 by GDP PPP (WDI, constant 2021 USD, year 2022)
gdp_col = :wdi_gdppppcon2021
gdp_df = select(dropmissing(qog_recent, gdp_col), :ident_ccodealp, :ident_cname, gdp_col)
sort!(gdp_df, gdp_col, rev=true)
top30_gdp = first(gdp_df, 30)

println("Top 30 by GDP PPP (2022):")
println("─"^110)
for r in eachrow(top30_gdp)
    m = filter(row -> row.ident_ccodealp == r.ident_ccodealp, master)
    nrow(m) == 0 && continue
    mr = first(m)
    missing_flags = join([replace(string(col), "has_" => "") for col in [:has_emdat, :has_dose, :has_shdi, :has_geodist, :has_gravity, :has_lv] if !mr[col]], ", ")
    gdp_t = round(r[gdp_col] / 1e12, digits=2)
    println("  $(rpad(r.ident_ccodealp, 5)) $(rpad(r.ident_cname, 40)) $(lpad(string(gdp_t), 7))T $(isempty(missing_flags) ? "✓ all" : "MISSING: $missing_flags")")
end


Top 30 by GDP PPP (2022):
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  CHN   China                                      29.68T ✓ all
  USA   United States of America (the)             24.05T ✓ all
  IND   India                                      12.18T ✓ all
  JPN   Japan                                       5.65T ✓ all
  RUS   Russian Federation (the)                    5.61T ✓ all
  DEU   Germany                                     5.25T ✓ all
  BRA   Brazil                                       3.9T ✓ all
  FRA   France                                      3.74T ✓ all
  IDN   Indonesia                                   3.72T ✓ all
  GBR   United Kingdom of Great Britain and Northern Ireland (the)     3.7T ✓ all
  ITA   Italy                                       3.07T ✓ all
  TUR   Turkey                                      2.81T ✓ all
  MEX   Mexico                                      2.78T ✓ all
  KOR   Korea

In [20]:
# Saudi Arabia subnational coverage in SHDI
sau_shdi = filter(r -> r.isocode3 == "SAU", shdi)
sau_sub = filter(r -> r.level == "Subnat", sau_shdi)
println("SAU in SHDI: $(nrow(sau_sub)) subnational rows")
println("Regions: $(length(unique(sau_sub.gdlcode)))")
println("Years: $(minimum(sau_sub.year))–$(maximum(sau_sub.year))")
println()
# List the regions
sau_2020 = filter(r -> r.year == 2020, sau_sub)
for r in eachrow(select(sau_2020, :gdlcode, :region))
    println("  $(r.gdlcode) — $(r.region)")
end


SAU in SHDI: 170 subnational rows
Regions: 5
Years: 1990–2023

  SAUr101 — Center (Riadh, Qassim)
  SAUr102 — West (Makka, Madinah, Tabuk)
  SAUr103 — South (Bahah, Jizan, Asir, Najran)
  SAUr104 — Eastern province
  SAUr105 — North (Northern Borders, Al Jawf, Hail)


In [21]:
# SAU subnational HDI and income — is there any variation?
sau_2020 = filter(r -> r.isocode3 == "SAU" && r.year == 2020 && r.level == "Subnat", shdi)
println("SAU subnational HDI (2020):")
for r in eachrow(select(sau_2020, :region, :shdi, :healthindex, :edindex, :incindex))
    println("  $(rpad(r.region, 40)) HDI=$(round(r.shdi, digits=3))  health=$(round(r.healthindex, digits=3))  ed=$(round(r.edindex, digits=3))  inc=$(round(r.incindex, digits=3))")
end
println()
println("CV of SHDI: $(round(std(sau_2020.shdi) / mean(sau_2020.shdi), digits=4))")
println("CV of income index: $(round(std(sau_2020.incindex) / mean(sau_2020.incindex), digits=4))")


SAU subnational HDI (2020):
  Center (Riadh, Qassim)                   HDI=0.921  health=0.886  ed=0.935  inc=0.942
  West (Makka, Madinah, Tabuk)             HDI=0.892  health=0.886  ed=0.885  inc=0.903
  South (Bahah, Jizan, Asir, Najran)       HDI=0.875  health=0.886  ed=0.811  inc=0.932
  Eastern province                         HDI=0.883  health=0.886  ed=0.827  inc=0.938
  North (Northern Borders, Al Jawf, Hail)  HDI=0.91  health=0.886  ed=0.876  inc=0.972

CV of SHDI: 0.0212
CV of income index: 0.0263


## 4. Phase 1 Status Extraction

Extract per-country status from the row-level `country_missingness_flags.csv`.

In [23]:
using StatsBase

In [24]:
# Load missingness flags — keyed by ggis_rowid, need to join to QoG to get country
miss_flags = CSV.read("data/country_missingness_flags.csv", DataFrame)
println("Missingness flags: $(nrow(miss_flags)) rows")
println("Columns: $(names(miss_flags))")
println()

# Join to QoG to get country code per row
# ggis_rowid is 1-indexed row number in the augmented QoG
qog_with_status = hcat(
    select(qog, :ident_ccodealp, :ident_year),
    select(miss_flags, :ggis_country_status, :ggis_global_slug_coverage)
)

# Extract per-country summary
country_status = combine(groupby(dropmissing(qog_with_status, :ident_ccodealp), :ident_ccodealp),
    :ggis_country_status => (x -> argmax(countmap(x))) => :phase1_modal_status,
    :ggis_country_status => last => :phase1_latest_status,
    :ggis_global_slug_coverage => mean => :mean_slug_coverage,
    :ggis_country_status => (x -> length(unique(x))) => :n_distinct_statuses
)

println("Country statuses: $(nrow(country_status)) countries")
println()
println("Modal status distribution:")
combine(groupby(country_status, :phase1_modal_status), nrow => :n) |> df -> sort(df, :n, rev=true)

Missingness flags: 12391 rows
Columns: ["ggis_rowid", "ggis_country_status", "ggis_global_slug_coverage", "ggis_peer_deviation", "ggis_lag_affected"]

Country statuses: 202 countries

Modal status distribution:


Row,phase1_modal_status,n
,String31,Int64
1,strong,130
2,reporting,54
3,microstate,11
4,self_exclusion,2
5,failed,2
6,political_exclusion,1
7,nascent,1
8,collision,1


In [26]:
# Join Phase 1 status to master
master = leftjoin(master, country_status, on=:ident_ccodealp)
println("Master with Phase 1 status: $(nrow(master)) rows × $(ncol(master)) cols")
println("Countries with status: $(count(!ismissing, master.phase1_modal_status))")

Master with Phase 1 status: 202 rows × 27 cols
Countries with status: 202


In [27]:
# How does Phase 1 status predict external dataset coverage?
status_coverage = combine(groupby(master, :phase1_modal_status),
    nrow => :n_countries,
    :has_emdat => mean => :pct_emdat,
    :has_dose => mean => :pct_dose,
    :has_shdi => mean => :pct_shdi,
    :has_geodist => mean => :pct_geodist,
    :has_gravity => mean => :pct_gravity,
    :has_lv => mean => :pct_lv
)

# Format as percentages
for col in [:pct_emdat, :pct_dose, :pct_shdi, :pct_geodist, :pct_gravity, :pct_lv]
    status_coverage[!, col] = round.(100 .* status_coverage[!, col], digits=0)
end

sort!(status_coverage, :n_countries, rev=true)
println("External dataset coverage by Phase 1 status (% of countries in each status):")
status_coverage


External dataset coverage by Phase 1 status (% of countries in each status):


Row,phase1_modal_status,n_countries,pct_emdat,pct_dose,pct_shdi,pct_geodist,pct_gravity,pct_lv
,String31?,Int64,Float64,Float64,Float64,Float64,Float64,Float64
1,strong,130,100.0,56.0,99.0,99.0,100.0,77.0
2,reporting,54,100.0,17.0,94.0,93.0,100.0,33.0
3,microstate,11,64.0,0.0,55.0,82.0,100.0,0.0
4,self_exclusion,2,100.0,0.0,0.0,50.0,100.0,0.0
5,failed,2,100.0,0.0,0.0,0.0,100.0,0.0
6,political_exclusion,1,100.0,0.0,0.0,100.0,100.0,0.0
7,nascent,1,0.0,0.0,0.0,0.0,0.0,0.0
8,collision,1,0.0,0.0,0.0,0.0,100.0,0.0


## 5. Verify Master Reference

Final checks before writing Arrow.

In [28]:
# Final verification
println("Master country reference: $(nrow(master)) countries × $(ncol(master)) columns")
println()
println("Column completeness:")
for col in names(master)
    n_present = count(!ismissing, master[!, col])
    pct = round(100 * n_present / nrow(master), digits=1)
    println("  $(rpad(string(col), 25)) $n_present / $(nrow(master))  ($pct%)")
end
println()
println("Countries with ALL 6 external datasets: $(count(r -> r.has_emdat && r.has_dose && r.has_shdi && r.has_geodist && r.has_gravity && r.has_lv, eachrow(master)))")
println("Countries with NO external datasets: $(count(r -> !r.has_emdat && !r.has_dose && !r.has_shdi && !r.has_geodist && !r.has_gravity && !r.has_lv, eachrow(master)))")

Master country reference: 202 countries × 27 columns

Column completeness:
  ident_ccodealp            202 / 202  (100.0%)
  ident_cname               202 / 202  (100.0%)
  ident_ccode               202 / 202  (100.0%)
  qog_first_year            202 / 202  (100.0%)
  qog_last_year             202 / 202  (100.0%)
  qog_n_years               202 / 202  (100.0%)
  un_region                 198 / 202  (98.0%)
  un_subregion              198 / 202  (98.0%)
  continent                 198 / 202  (98.0%)
  has_emdat                 202 / 202  (100.0%)
  has_dose                  202 / 202  (100.0%)
  has_shdi                  202 / 202  (100.0%)
  has_geodist               202 / 202  (100.0%)
  has_gravity               202 / 202  (100.0%)
  has_lv                    202 / 202  (100.0%)
  emdat_first_year          196 / 202  (97.0%)
  emdat_last_year           196 / 202  (97.0%)
  dose_first_year           82 / 202  (40.6%)
  dose_last_year            82 / 202  (40.6%)
  shdi_first_year     

In [29]:
# Who has NO external datasets?
println("Countries with NO external datasets:")
for r in eachrow(filter(r -> !r.has_emdat && !r.has_dose && !r.has_shdi && !r.has_geodist && !r.has_gravity && !r.has_lv, master))
    println("  $(r.ident_ccodealp) — $(r.ident_cname) (status: $(r.phase1_modal_status))")
end


Countries with NO external datasets:
  XTI — Tibet (status: nascent)


In [30]:
# Countries in external datasets but NOT in QoG — are any significant?
all_external = union(
    Set(unique(emdat_cy.iso3)),
    Set(unique(dose_sub.iso3)),
    Set(unique(shdi.isocode3)),
    Set(unique(skipmissing(geodist.iso_o))),
    Set(unique(skipmissing(gravity.iso3_o))),
    Set(unique(skipmissing(lv_panel.iso3)))
)

external_only = sort(collect(setdiff(all_external, qog_codes)))
println("Countries in external datasets but NOT in QoG ($(length(external_only))):")
println()

# Check which ones are territories vs real states
for c in external_only
    # Find which datasets have it
    sources = String[]
    c in Set(unique(emdat_cy.iso3)) && push!(sources, "emdat")
    c in Set(unique(dose_sub.iso3)) && push!(sources, "dose")
    c in Set(unique(shdi.isocode3)) && push!(sources, "shdi")
    c in Set(unique(skipmissing(geodist.iso_o))) && push!(sources, "geodist")
    c in Set(unique(skipmissing(gravity.iso3_o))) && push!(sources, "gravity")
    c in Set(unique(skipmissing(lv_panel.iso3))) && push!(sources, "lv")
    println("  $(rpad(c, 5)) sources: $(join(sources, ", "))")
end

Countries in external datasets but NOT in QoG (51):

  ABW   sources: geodist, gravity
  AIA   sources: emdat, geodist, gravity
  ANT   sources: emdat, dose, geodist, gravity
  ASM   sources: emdat, gravity
  AZO   sources: emdat
  BES   sources: gravity
  BLM   sources: emdat
  BMU   sources: emdat, geodist, gravity
  CCK   sources: geodist, gravity
  COK   sources: emdat, geodist, gravity
  CUW   sources: emdat, gravity
  CXR   sources: geodist, gravity
  CYM   sources: emdat, geodist, gravity
  DFR   sources: emdat
  ESH   sources: geodist, gravity
  FLK   sources: geodist, gravity
  FRO   sources: geodist, gravity
  GIB   sources: geodist, gravity
  GLP   sources: emdat, geodist, gravity
  GRL   sources: geodist, gravity
  GUF   sources: emdat, geodist, gravity
  GUM   sources: emdat, gravity
  HKG   sources: emdat, geodist, gravity
  IOT   sources: gravity
  MAC   sources: emdat, geodist, gravity
  MAF   sources: emdat
  MNP   sources: emdat, geodist, gravity
  MSR   sources: emda

In [31]:
# ## Write Arrow
# for col in names(master)
#     if nonmissingtype(eltype(master[!, col])) <: AbstractString
#         master[!, col] = passmissing(String).(master[!, col])
#     end
# end

# Arrow.write("data/ggis_country_master.arrow", master)
# println("  → data/ggis_country_master.arrow  ($(round(filesize("data/ggis_country_master.arrow") / 1024, digits=1)) KB)")


  → data/ggis_country_master.arrow  (44.2 KB)
